# PyTorch 입문 — Compose · MNIST · DataLoader

`__call__` transform, `Compose`, MNIST, 배치 shape, 한 장 그리기를 연습합니다.  
이론 복습: [`../session05.md`](../session05.md)

> **이 노트북이 하는 일**  
> `adder`의 `__call__` → `Compose` → MNIST(`CenterCrop`+`ToTensor`) → shape → DataLoader → `imshow`.

환경: `torch-env` 커널을 쓰는 것을 권장합니다.  
첫 실행 시 MNIST 다운로드에 네트워크가 필요합니다.

## 1. import

`torchvision.transforms`로 전처리 파이프라인을 만듭니다.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

## 2. session04 복습 — hi_set · scaling

Dataset 본편은 session04입니다. 여기서는 `__call__`·Compose 연습용으로만 둡니다.

In [ ]:
class hi_set(Dataset):
    def __init__(self, length=10, transform=None):
        torch.manual_seed(101)
        self.x = 10 * torch.ones(length, 2)
        self.y = torch.ones(length, 1)
        self.transform = transform
        self.len = length

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        sample_data = self.x[idx], self.y[idx]
        if self.transform:
            sample_data = self.transform(sample_data)
        return sample_data


def scaling(sample):
    x, y = sample
    return x / 10.0, y / 10.0

## 3. `__call__` — adder

객체를 함수처럼 쓰면 `__call__`이 자동 실행됩니다.

In [ ]:
class adder:
    def __init__(self, add1=1):
        self.add1 = add1

    def __call__(self, sample):
        # sample은 (x, y) 쌍. 둘 다 add1만큼 더한 쌍을 돌려준다.
        x, y = sample
        return x + self.add1, y + self.add1


a1 = adder()
demo = hi_set(10, transform=a1)
print(demo[-1])

## 4. Compose — scaling 다음 adder

리스트 **앞**이 먼저 실행됩니다.

In [ ]:
data_transform = transforms.Compose([scaling, a1])
composed = hi_set(3, transform=data_transform)
print(next(iter(composed)))

## 5. MNIST — CenterCrop(20) · ToTensor

원본 28×28 가운데를 20×20으로 자르고, 텐서로 바꿉니다.

In [ ]:
crop_trans = transforms.Compose([
    transforms.CenterCrop(20),
    transforms.ToTensor(),
])

train_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=crop_trans,
)
test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=crop_trans,
)

img0, label0 = train_data[0]
print("한 장 shape:", img0.shape)  # (C, H, W) = (1, 20, 20)
print("정답:", label0)

## 6. 한 장 그리기

`(1, 20, 20)`을 2차원으로 펼쳐 `imshow`합니다.

In [ ]:
plt.imshow(img0.view(20, 20), cmap="gray")
plt.title(label0)
plt.show()

## 7. DataLoader — batch_size=64

배치 shape은 `(B, C, H, W)`입니다. `squeeze`로 채널 축을 접어 그립니다.

In [ ]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

train_features, train_labels = next(iter(train_loader))
print("배치 shape:", train_features.shape)  # (64, 1, 20, 20)
print("한 장 shape:", train_features[0].shape)
print("정답 묶음 shape:", train_labels.shape)

f1 = train_features[0].squeeze()
l1 = train_labels[0]
plt.imshow(f1, cmap="gray")
plt.title(int(l1))
plt.show()

---

이론 복습: [`../session05.md`](../session05.md) · Dataset 복습: [`04_custom_dataset.ipynb`](./04_custom_dataset.ipynb)